In [1]:
!pip install databento

In [3]:
import databento as db

In [4]:
import databento as db
from databento import DBNStore, Dataset, Schema

import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment
DB_API_KEY = os.getenv('DATABENTO_API_KEY')
print('Loaded API key:', '***' if DB_API_KEY else 'NOT FOUND')

# Initialize client
client = db.Historical(DB_API_KEY)

Loaded API key: ***


In [5]:
data = client.timeseries.get_range(
    dataset="XNAS.ITCH",
    symbols=["SPY"],
    schema="ohlcv-1m",
    start="2019-01-01T00:00:00",
    end="2025-07-10T00:00:00"
)

C:\Users\Tom\AppData\Local\Temp\ipykernel_12068\1827881145.py:1: BentoWarning: The streaming request contained one or more days which have reduced quality: 2021-07-07 (degraded), 2021-10-26 (degraded), 2022-09-19 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  data = client.timeseries.get_range(


In [ ]:
data.to_file('spy_ohlcv_20190102_20250710.dbn')

<DBNStore(schema=ohlcv-1m)>

In [7]:
data.to_df().head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2019-01-02 09:00:00+00:00,33,2,7294,245.38,245.43,245.08,245.08,1655,SPY
2019-01-02 09:01:00+00:00,33,2,7294,245.01,245.19,245.01,245.19,7055,SPY
2019-01-02 09:02:00+00:00,33,2,7294,245.19,245.24,244.96,245.24,964,SPY
2019-01-02 09:03:00+00:00,33,2,7294,245.22,245.22,245.22,245.22,8,SPY
2019-01-02 09:04:00+00:00,33,2,7294,245.26,245.40,245.26,245.40,389,SPY


In [ ]:
# Save the DBNStore object to a .dbn file


In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from datetime import datetime, time
import pytz

def plot_interactive_ohlc(df, start_date, end_date=None, or_start_time="09:30:00", or_end_time="10:15:00"):
    """
    Create an interactive OHLC plot with opening range analysis.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with OHLC data (columns: open, high, low, close, volume)
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str, optional
        End date in 'YYYY-MM-DD' format. If None, uses only start_date
    or_start_time : str
        Opening range start time in 'HH:MM:SS' format (default: "09:30:00")
    or_end_time : str
        Opening range end time in 'HH:MM:SS' format (default: "10:15:00")
    
    Returns:
    --------
    plotly.graph_objects.Figure
        Interactive plotly figure
    """
    
    # Filter data by date range
    if end_date is None:
        # Single day
        filtered_df = df.loc[start_date]
        title_date = start_date
    else:
        # Date range
        filtered_df = df.loc[start_date:end_date]
        title_date = f"{start_date} to {end_date}"
    
    if filtered_df.empty:
        print(f"No data found for the specified date range: {title_date}")
        return None
    
    # Create subplots with secondary y-axis for volume
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=(f'SPY OHLC - {title_date}', 'Volume'),
        row_heights=[0.7, 0.3]
    )
    
    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=filtered_df.index,
            open=filtered_df['open'],
            high=filtered_df['high'],
            low=filtered_df['low'],
            close=filtered_df['close'],
            name='SPY',
            hovertemplate='<b>%{x}</b><br>' +
                         'Open: $%{open:.2f}<br>' +
                         'High: $%{high:.2f}<br>' +
                         'Low: $%{low:.2f}<br>' +
                         'Close: $%{close:.2f}<br>' +
                         '<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Add volume bars
    fig.add_trace(
        go.Bar(
            x=filtered_df.index,
            y=filtered_df['volume'],
            name='Volume',
            marker_color='rgba(158,158,158,0.8)',
            hovertemplate='<b>%{x}</b><br>' +
                         'Volume: %{y:,}<br>' +
                         '<extra></extra>'
        ),
        row=2, col=1
    )
    
    # Calculate Opening Range for each day in the dataset
    if end_date is None:
        # Single day analysis
        days_to_analyze = [start_date]
    else:
        # Multi-day analysis - get unique dates
        days_to_analyze = filtered_df.index.date
        days_to_analyze = sorted(list(set(days_to_analyze)))
        days_to_analyze = [str(day) for day in days_to_analyze]
    
    # Add OR analysis for each day
    colors = ['red', 'green', 'blue', 'orange', 'purple']  # Cycle through colors for multiple days
    
    for i, day in enumerate(days_to_analyze):
        try:
            # Get day data
            day_data = df.loc[day]
            if day_data.empty:
                continue
                
            # Create OR time range
            timezone = day_data.index.tz if hasattr(day_data.index, 'tz') else None
            or_start = pd.Timestamp(f"{day} {or_start_time}", tz=timezone)
            or_end = pd.Timestamp(f"{day} {or_end_time}", tz=timezone)
            
            # Get OR data
            or_data = day_data.loc[or_start:or_end]
            if or_data.empty:
                continue
                
            or_high = or_data['high'].max()
            or_low = or_data['low'].min()
            
            # Choose colors
            high_color = colors[i % len(colors)]
            low_color = colors[(i + 1) % len(colors)]
            
            # Add horizontal lines for OR high and low
            fig.add_hline(
                y=or_high,
                line_dash="dash",
                line_color=high_color,
                line_width=2,
                annotation_text=f"OR High {day}: ${or_high:.2f}",
                annotation_position="top right",
                row=1
            )
            
            fig.add_hline(
                y=or_low,
                line_dash="dash", 
                line_color=low_color,
                line_width=2,
                annotation_text=f"OR Low {day}: ${or_low:.2f}",
                annotation_position="bottom right",
                row=1
            )
            
            # Add vertical lines for OR start and end
            y_min = day_data['low'].min() * 0.998
            y_max = day_data['high'].max() * 1.002
            
            # OR Start line
            fig.add_shape(
                type="line",
                x0=or_start, x1=or_start,
                y0=y_min, y1=y_max,
                line=dict(color="blue", width=2, dash="solid"),
                row=1, col=1
            )
            
            # OR End line
            fig.add_shape(
                type="line",
                x0=or_end, x1=or_end,
                y0=y_min, y1=y_max,
                line=dict(color="purple", width=2, dash="solid"),
                row=1, col=1
            )
            
            # Add shaded box for OR time period
            fig.add_vrect(
                x0=or_start,
                x1=or_end,
                fillcolor="rgba(255, 255, 0, 0.1)",
                layer="below",
                line_width=0,
                annotation_text=f"OR {day}",
                annotation_position="top left",
                row=1
            )
            
            # Print OR analysis
            print(f"📊 Opening Range Analysis for {day}:")
            print(f"   OR High: ${or_high:.2f}")
            print(f"   OR Low: ${or_low:.2f}")
            print(f"   OR Size: ${or_high - or_low:.2f}")
            print(f"   OR Period: {or_start.strftime('%H:%M')} to {or_end.strftime('%H:%M')}")
            print(f"   Bars in OR: {len(or_data)}")
            print()
            
        except Exception as e:
            print(f"Could not analyze OR for {day}: {e}")
            continue
    
    # Update layout for better interactivity
    fig.update_layout(
        title=f'Interactive SPY Chart - {title_date}',
        height=800,
        xaxis_rangeslider_visible=False,
        hovermode='x unified',
        showlegend=True
    )
    
    # Update x-axis for better time display
    fig.update_xaxes(
        title="Time",
        tickformat="%H:%M" if end_date is None else "%m-%d %H:%M",
        row=2, col=1
    )
    
    # Update y-axes
    fig.update_yaxes(title="Price ($)", row=1, col=1)
    fig.update_yaxes(title="Volume", row=2, col=1)
    
    return fig

# Example usage:
# fig = plot_interactive_ohlc(spy_df, '2025-06-18')
# fig.show()

In [8]:
# Load and prepare the data first
spy_data = db.DBNStore.from_file('spy_ohlcv_20190102_20250710.dbn')
spy_df = spy_data.to_df()

# Apply timezone conversion to US/Eastern
import pytz
if spy_df.index.tz is not None:
    eastern = pytz.timezone('US/Eastern')
    spy_df.index = spy_df.index.tz_convert(eastern)

print(f"Data loaded: {len(spy_df)} bars")
print(f"Date range: {spy_df.index.min()} to {spy_df.index.max()}")
print(f"Timezone: {spy_df.index.tz}")

Data loaded: 1217772 bars
Date range: 2019-01-02 04:00:00-05:00 to 2025-07-09 19:56:00-04:00
Timezone: US/Eastern


In [9]:
# Example 1: Plot a single day with default OR times (9:30-10:15)
fig1 = plot_interactive_ohlc(spy_df, '2025-06-18')
fig1.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.Candlestick: 'hovertemplate'

Did you mean "hovertext"?

    Valid properties:
        close
            Sets the close values.
        closesrc
            Sets the source reference on Chart Studio Cloud for
            `close`.
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
            that, "scatter" traces also appends customdata items in
            the markers DOM elements
        customdatasrc
            Sets the source reference on Chart Studio Cloud for
            `customdata`.
        decreasing
            :class:`plotly.graph_objects.candlestick.Decreasing`
            instance or dict with compatible properties
        high
            Sets the high values.
        highsrc
            Sets the source reference on Chart Studio Cloud for
            `high`.
        hoverinfo
            Determine

In [ ]:
# Example 2: Plot with custom opening range times (9:30-10:00)
fig2 = plot_interactive_ohlc(spy_df, '2025-06-18', or_start_time="09:30:00", or_end_time="10:00:00")
fig2.show()

In [ ]:
# Example 3: Plot multiple days (will show OR for each day)
fig3 = plot_interactive_ohlc(spy_df, '2025-06-18', '2025-06-20')
fig3.show()